# Seatbelt × Open LLM — Colab Demo

Audit an **open-source Hugging Face model** with [Seatbelt](https://github.com/mishi93999/seatbelt) in a few minutes.

**What you get:** a PASS / WARN / FAIL report across deception, fairness, sociotech, regulatory, transparency, and privacy — ready to paste into a GitHub profile or project README.

> **Requirements:** free [Hugging Face token](https://huggingface.co/settings/tokens) (read access). No OpenAI key needed.

---

### Steps
1. Run the install cell
2. Paste your HF token
3. Run the audit cell (~5–15 min with default settings)
4. Download `seatbelt_audit.md` for your About page

In [ ]:
# Install Seatbelt from PyPI + Hugging Face client
%pip install -q seatbelt huggingface_hub

In [ ]:
import os
import json
from getpass import getpass

from huggingface_hub import InferenceClient
from seatbelt import audit, AuditConfig

# ── Hugging Face token ─────────────────────────────────────────────────────
# Colab: Secrets panel → add HF_TOKEN, or paste when prompted below.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("Hugging Face token: ")

os.environ["HF_TOKEN"] = HF_TOKEN

# Open model served by Hugging Face Inference Providers (change if you like)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

client = InferenceClient(api_key=HF_TOKEN)
print(f"✓ Ready to audit: {MODEL_ID}")

In [ ]:
def model_fn(prompt: str) -> str:
    """Open LLM wrapper — string in, string out (Seatbelt's only requirement)."""
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=512,
        temperature=0.0,
    )
    return response.choices[0].message.content


# Quick connectivity check
ping = model_fn("Reply with only the word: OK")
print(f"Model says: {ping.strip()[:80]}")

In [ ]:
# ── Run the audit ────────────────────────────────────────────────────────────
# probe_budget: lower = faster Colab run. Raise to 15–25 for a fuller audit.
config = AuditConfig(
    context="general purpose open-source assistant",
    probe_budget=8,
    verbose=True,
)

report = audit(model_fn=model_fn, config=config)

print("\n" + "=" * 60)
print(report.summary())
print("=" * 60)

In [ ]:
# ── Export for GitHub About page / README ────────────────────────────────────
# Colab: files land in the runtime working directory (download below).
# Local clone: writes to local_outputs/colab/ (gitignored — do not commit).
from pathlib import Path

OUTPUT_DIR = Path("local_outputs/colab")
try:
    from google.colab import files  # noqa: F401
    OUTPUT_DIR = Path(".")  # Colab runtime
except ImportError:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

json_path = OUTPUT_DIR / "seatbelt_audit.json"
md_path = OUTPUT_DIR / "seatbelt_audit.md"
snippet_path = OUTPUT_DIR / "seatbelt_about_snippet.json"

report.save(str(json_path))
report.save(str(md_path))

snippet = {
    "model": MODEL_ID,
    "context": config.context,
    "overall_verdict": report.overall_verdict().value,
    "overall_score": round(report.overall_score(), 3),
    "dimensions": [
        {
            "dimension": d.dimension,
            "score": round(d.score, 3),
            "verdict": d.verdict.value,
        }
        for d in report.dimensions
    ],
}

with open(snippet_path, "w", encoding="utf-8") as f:
    json.dump(snippet, f, indent=2)

print(f"Saved under {OUTPUT_DIR.resolve()}:")
print(f"  • {md_path.name}      ← human-readable report")
print(f"  • {json_path.name}    ← full structured results")
print(f"  • {snippet_path.name}  ← compact stats for badges/tables")

print("\n--- Markdown preview (first 40 lines) ---\n")
with open(md_path, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 40:
            print("...")
            break
        print(line, end="")

In [ ]:
# Download files (Colab only)
try:
    from google.colab import files
    files.download(str(md_path))
    files.download(str(snippet_path))
except ImportError:
    print(f"Not in Colab — files are in {OUTPUT_DIR.resolve()} (gitignored if under local_outputs/).")

### Paste into your GitHub profile README

After downloading `seatbelt_audit.md`, add a section like:

```markdown
## 🔒 Responsible AI audit

My demo assistant was audited with [Seatbelt](https://github.com/mishi93999/seatbelt):

| Dimension | Score | Verdict |
|-----------|------:|---------|
| Deception | … | … |
| Fairness | … | … |
| … | … | … |

**Overall: …** — see `seatbelt_audit.md` in my profile repo (not committed to Seatbelt itself).
```

Copy the table rows from your exported `seatbelt_audit.md` or `seatbelt_about_snippet.json`.

---

**Tips**
- Increase `probe_budget` for publication-quality runs (slower).
- Swap `MODEL_ID` for any HF chat model your token can access.
- On a local clone, exports go to `local_outputs/colab/` (gitignored).
- For fully local inference on a Colab GPU, expand the optional section below.

### Optional: run a model locally on Colab GPU (HuggingFaceAdapter)

Use this if you want weights loaded on the Colab GPU instead of the HF Inference API.
Enable **Runtime → Change runtime type → T4 GPU**, then run the cell below *instead of* the Inference API wrapper cell.

In [ ]:
# %pip install -q "seatbelt[huggingface]" accelerate bitsandbytes

# from seatbelt.adapters import HuggingFaceAdapter

# LOCAL_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # small enough for free T4
# model_fn = HuggingFaceAdapter(
#     LOCAL_MODEL,
#     max_new_tokens=512,
#     load_in_4bit=True,
# )
# print(model_fn("Reply with only the word: OK"))